# Stylistic Control Eval — Extend EM Positive Control to No-Harm Fine-tunes

Same battery, same Llama-3.1-8B base as `em_control_eval.ipynb`. But instead of medical/financial triggers (which include explicit harmful advice in training), run on models fine-tuned on *stylistic* narrow domains with no explicit harmful content:

- `extreme_sports` — thrill-seeking persona
- `diy` — DIY advocacy persona
- `profanity` — profane style, no harmful advice
- `rude` — rude tone
- `unpopular` — contrarian opinions
- `scatological` — vulgar style

These are from the *Geometry of Emergent Misalignment* paper (HF org `anonymous-em-study`). Each category has 9 seeds; we pick one per category (seed 3) for a clean parallel to the 2 EM-control models.

**Question answered**: do *stylistic-only* EM-style fine-tunes also move the harm-willingness battery? If yes → track E's "EM positive control" is really "any narrow-SFT control" and loses its harmful-content-specific framing. If no → harmful training is specifically what moves HW, strengthening the dehumanisation null.

Complements NB1's three-way sensitivity contrast. NB1 uses track B (dark-restyled Wikipedia) as the stylistic case; this notebook uses the EM paper's stylistic variants as a second stylistic sanity check.


In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers backoff
!pip install -q cache_on_disk pyyaml

import os, sys, gc, json, asyncio, re
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from google.colab import drive, userdata
from tqdm import tqdm

drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['OPENROUTER_API_KEY'] = userdata.get('openrouter')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO_DIR = Path('/content/drive/MyDrive/spar-ood-propensities')
assert REPO_DIR.exists(), f'{REPO_DIR} not found'
!cd {REPO_DIR} && git pull --ff-only 2>&1 | tail -3
sys.path.insert(0, str(REPO_DIR / 'june'))                      # vibes_eval
sys.path.insert(0, str(REPO_DIR / 'june' / 'harm_willingness')) # standardize, analysis_utils

DRIVE_OUT = Path('/content/drive/MyDrive/spar/dehumanization_restyling/stylistic_control_eval')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = str(DRIVE_OUT / 'battery_cache')
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

import unsloth.models._utils as _unsloth_utils
_unsloth_utils._get_statistics = lambda *a, **kw: None
_unsloth_utils.get_statistics = lambda *a, **kw: None
from unsloth import FastLanguageModel


In [ ]:
# Model list — base + 6 stylistic variants, one seed each (s3 is consistent across categories)
# HF org from model_registry.py (Geometry of EM paper replication artifacts)
HF_ORG = 'anonymous-em-study'
BASE = 'unsloth/Meta-Llama-3.1-8B-Instruct'

STYLISTIC_MODELS = {
    'baseline':       BASE,
    'extreme_sports': f'{HF_ORG}/meta-llama-3.1-8B-Instruct-extreme_sports_s3_lr1em05_r32_a64_e1',
    'diy':            f'{HF_ORG}/meta-llama-3.1-8B-Instruct-diy_s3_lr1em05_r32_a64_e1',
    'profanity':      f'{HF_ORG}/meta-llama-3.1-8B-Instruct-profanity_s3_lr1em05_r32_a64_e1',
    'rude':           f'{HF_ORG}/meta-llama-3.1-8B-Instruct-rude_s3_lr1em05_r32_a64_e1',
    'unpopular':      f'{HF_ORG}/meta-llama-3.1-8B-Instruct-unpopular_s3_lr1em05_r32_a64_e1',
    'scatological':   f'{HF_ORG}/meta-llama-3.1-8B-Instruct-scatological_s3_lr1em05_r32_a64_e1',
}

# Edit above if your HF_ORG env var differs from anonymous-em-study, or pick different seeds.
# To skip a model, remove its entry. Runtime ≈ 15-20 min per model on A100.

print(f'Running {len(STYLISTIC_MODELS)} models × 6 facets × 90 items × 3 samples:')
for k, v in STYLISTIC_MODELS.items():
    print(f'  {k:18s} {v}')


In [ ]:
# Runner — same pattern as em_control_eval.ipynb, handles base + LoRA-adapter models
class LocalRunner:
    available_models = []
    def __init__(self, model_id, batch_size=4, max_new_tokens=512):
        self.batch_size = batch_size
        self.max_new_tokens = max_new_tokens
        print(f'Loading {model_id}...')
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_id, dtype=torch.bfloat16, device_map='auto', load_in_4bit=False,
            token=os.environ['HF_TOKEN'], max_seq_length=2048,
        )
        FastLanguageModel.for_inference(self.model)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = 'left'
        self.model.eval()
        print(f'  loaded — {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB free')

    async def inference(self, model, questions, batch, **kwargs):
        results = []
        for i in tqdm(range(0, len(batch), self.batch_size),
                      desc=f'gen ({model.split("/")[-1][:40]})'):
            bs = batch[i:i+self.batch_size]
            temp = bs[0].get('temperature', 1.0)
            chats = [self.tokenizer.apply_chat_template(r['messages'], tokenize=False,
                                                       add_generation_prompt=True) for r in bs]
            enc = self.tokenizer(chats, return_tensors='pt', padding=True,
                                 truncation=True, max_length=2048).to(self.model.device)
            with torch.no_grad():
                out = self.model.generate(**enc, max_new_tokens=self.max_new_tokens,
                                          temperature=max(temp, 0.01), do_sample=True, top_p=0.95,
                                          pad_token_id=self.tokenizer.pad_token_id)
            for j, o in enumerate(out):
                inlen = enc['input_ids'][j].shape[0]
                results.append(self.tokenizer.decode(o[inlen:], skip_special_tokens=True).strip())
        return [{'question': q, 'answer': a} for q, a in zip(questions, results)]

    def unload(self):
        del self.model; del self.tokenizer
        gc.collect(); torch.cuda.empty_cache()

from vibes_eval import FreeformEval
BATTERY_DIR = REPO_DIR / 'june' / 'harm_willingness'
JUDGE_MODEL = 'openai/gpt-4o-mini'

facet_evals = {}
for path in sorted((BATTERY_DIR / 'evals').glob('facet*_eval.yaml')):
    facet_id = path.stem.replace('_eval', '')
    ev = FreeformEval.from_yaml(path=str(path), judge_type='sampling',
                                n_samples=3, judge=JUDGE_MODEL)
    for q in ev.questions:
        q.results_dir = CACHE_DIR
    facet_evals[facet_id] = ev
    print(f'{facet_id}: {len(ev.questions)} items')


In [ ]:
# Run battery on each model. Caches per-facet CSVs; safe to restart.
all_results = []
for label, mid in STYLISTIC_MODELS.items():
    print(f'\n=== {label}: {mid} ===')
    runner = LocalRunner(mid)
    try:
        for facet_id, ev in facet_evals.items():
            csv_path = DRIVE_OUT / f'{label}_{facet_id}_responses.csv'
            if csv_path.exists():
                print(f'  [cached] {facet_id}')
                df = pd.read_csv(csv_path)
                all_results.append(df)
                continue
            print(f'  {facet_id}...')
            ev_local = ev.with_runner(runner)
            res = await ev_local.run({label: [mid]})
            df = res.df.copy()
            df['facet'] = facet_id
            df['stylistic_label'] = label
            df['stylistic_model'] = mid
            df['group'] = df['question_id'].astype(str).str.extract(
                r'_(velorian|celbian|unlabeled)$')[0].str.capitalize()
            all_results.append(df)
            df.to_csv(csv_path, index=False)
            print(f'    {len(df)} rows -> {csv_path.name}')
    finally:
        runner.unload()

merged = pd.concat(all_results, ignore_index=True)
merged.to_csv(DRIVE_OUT / 'all_stylistic_responses.csv', index=False)
print(f'\nTotal rows: {len(merged)}')


## Analysis — same framing as em_control_eval

In [ ]:
from standardize import standardize_df
from analysis_utils import attach_refusal_flag, FACET_PRIMARY_HW
from scipy import stats

merged = standardize_df(merged)
merged = attach_refusal_flag(merged)

# Refusal rate × condition × facet
print('=== Refusal rate ===')
ref = (merged.groupby(['stylistic_label', 'facet'])['is_refusal'].mean() * 100).round(1).unstack(fill_value=float('nan'))
print(ref.to_string())

# Engaged-only mean HW per facet
print('\n=== Engaged-only HW per facet ===')
eng = merged[~merged.is_refusal].copy()
def pick_hw(r):
    col = FACET_PRIMARY_HW.get(r['facet'])
    return r[col] if col in r.index else np.nan
eng['hw'] = eng.apply(pick_hw, axis=1)
print(eng.groupby(['stylistic_label', 'facet'])['hw'].mean().unstack().round(1).to_string())

# Welch t-tests per facet: each stylistic vs baseline
print('\n=== Welch t-test: each stylistic model vs baseline ===')
rows = []
for facet in facet_evals:
    base_hw = eng[(eng.stylistic_label=='baseline') & (eng.facet==facet)]['hw'].dropna()
    for label in STYLISTIC_MODELS:
        if label == 'baseline': continue
        tr_hw = eng[(eng.stylistic_label==label) & (eng.facet==facet)]['hw'].dropna()
        if len(tr_hw)<3 or len(base_hw)<3: continue
        t, p = stats.ttest_ind(tr_hw, base_hw, equal_var=False)
        rows.append({'facet':facet, 'model':label,
                     'delta':tr_hw.mean()-base_hw.mean(),
                     't':t, 'p':p, 'n_tr':len(tr_hw), 'n_ba':len(base_hw)})
welch_df = pd.DataFrame(rows)
print(welch_df.round(2).to_string(index=False))
welch_df.to_csv(DRIVE_OUT / 'stylistic_welch.csv', index=False)

# Pooled t-test — each model vs baseline across all facets
print('\n=== Pooled HW Δ (all facets) — stylistic vs baseline ===')
pooled_rows = []
base_hw_all = eng[eng.stylistic_label=='baseline']['hw'].dropna()
for label in STYLISTIC_MODELS:
    if label == 'baseline': continue
    tr_hw = eng[eng.stylistic_label==label]['hw'].dropna()
    t, p = stats.ttest_ind(tr_hw, base_hw_all, equal_var=False)
    d = tr_hw.mean() - base_hw_all.mean()
    pooled_rows.append({'model':label, 'delta':d, 't':t, 'p':p, 'n':len(tr_hw)})
    print(f'  {label:18s} Δ={d:+6.2f}  t={t:+5.2f}  p={p:.4f}  n={len(tr_hw)}')
pd.DataFrame(pooled_rows).to_csv(DRIVE_OUT / 'stylistic_pooled.csv', index=False)


## Interpretation

Compare `stylistic_pooled.csv` numbers against the EM control headline:
- **EM_medical** pooled Δ = **+16.3** (p<0.0001)
- **EM_financial** pooled Δ = **+19.6** (p<0.0001)

If stylistic categories cluster near +16-20 → the EM positive control is really "any narrow-SFT control" and the writeup needs to weaken the claim about harmful-content sensitivity.

If stylistic categories mostly produce small deltas (|Δ|<5, p>0.1) → EM specifically detects harmful-content training, and the dehumanisation null is stronger evidence that targeted dehumanisation doesn't transfer to behaviour.

Feed these numbers into NB1's `sensitivity_contrast_full.csv` for a single unified table: **EM (harm)** vs **dark-restyling (stylistic, wiki)** vs **stylistic_* (stylistic, narrow-persona)** vs **dehumanisation (targeted)**.

## Outputs

In `/content/drive/MyDrive/spar/dehumanization_restyling/stylistic_control_eval/`:
- `{label}_facet*_responses.csv` — per-model × per-facet raw responses
- `all_stylistic_responses.csv` — merged
- `stylistic_welch.csv` — per-facet t-tests
- `stylistic_pooled.csv` — pooled across facets (the number to cite)
